# SPARQL: запросы к графу знаний

**Курс «От интеллектуального поиска к интеллектуальным агентам», ВМК МГУ.** Туториал вводит основные понятия языка запросов SPARQL 1.1 на графе знаний романа «Евгений Онегин».

Роман уже знаком по занятию 5: там его 360 строф были корпусом для поиска по словам. Здесь из того же романа построен граф: герои, их родство и дружба, события, их порядок и причины, строфы и упоминания героев в строфах. На этом графе видно то, чего поиск по словам не умеет: идти по связям.

| шаг | понятие |
|---|---|
| 1 | граф из троек, запись Turtle |
| 2 | первый запрос: `SELECT`, `WHERE`, переменная |
| 3 | шаблон из нескольких троек и соединение |
| 4 | литералы, язык метки, `FILTER` |
| 5 | необязательная часть: `OPTIONAL` |
| 6 | альтернатива: `UNION`, `BIND` |
| 7 | отсутствие: `FILTER NOT EXISTS`, `MINUS`, открытый мир |
| 8 | счёт: `COUNT`, `GROUP BY`, `HAVING`, `ORDER BY`, `LIMIT` |
| 9 | пути по свойствам: `/`, `^`, `\|`, `+`, `*` |
| 10 | классы и подсвойства без машины вывода |
| 11 | окрестность узла на два шага, `VALUES` |
| 12 | `ASK` и `CONSTRUCT` |
| 13 | именованные графы и происхождение данных |
| 14 | граф и поиск по словам на одном вопросе |
| 15 | тот же приём на положении о рецензировании |
| 16 | точка доступа SPARQL |
| 17 | SPARQL как инструмент агента |

В конце упражнения с ответами и шпаргалка. Нужны `rdflib` и `pandas`, для шага 14 ещё `nltk`.

## Шаг 0. Граф «Онегина»

Граф собирает программа `build_graph.py`. У графа две части с разным происхождением:

* **ручная разметка** (`data/onegin_relations.ttl`): 11 персонажей, 3 места, 12 событий, родство и дружба, кто участвовал в событии, какое событие привело к какому, порядок событий, классы и свойства;
* **извлечено из текста** (`data/onegin_text.ttl`): 360 строф с номером главы, первой строкой и текстом, и тройки «герой упомянут в строфе». Упоминание находится по формам имени: «Татьяна», «Таня», «Тани» дают одного героя.

Обозначение строфы читается как «глава.строфа»: 4.XII есть глава 4, строфа XII. Письма Татьяны и Онегина обозначены 3.ПТ и 8.ПО.

In [1]:
import subprocess, sys
print(subprocess.run([sys.executable, "build_graph.py"], capture_output=True, text=True).stdout)

import pandas as pd
pd.set_option("display.max_colwidth", 70)
from onegin_kg import load, q, PREFIXES, G_REL, G_TXT, O
ds = load()
print("троек в ручной разметке:", len(ds.graph(G_REL)))
print("троек, извлечённых из текста:", len(ds.graph(G_TXT)))

ручная разметка: 269 троек | из текста: 2453 троек, строф 360, упоминаний 289



троек в ручной разметке: 269
троек, извлечённых из текста: 2453


## Шаг 1. Граф из троек

Граф RDF есть множество **троек** «субъект, предикат, объект». Субъект и предикат суть **IRI**, глобальные имена вида `http://example.org/onegin#Татьяна`. Объект бывает IRI или **литералом**: строкой, числом, датой. У строки бывает метка языка: `"Таня"@ru`.

Запись **Turtle** сокращает IRI через префикс: `:Татьяна` значит `http://example.org/onegin#Татьяна`. Слово `a` сокращает `rdf:type`. Точка с запятой продолжает тот же субъект, запятая продолжает тот же предикат.

Всё, что граф знает о Татьяне по ручной разметке:

In [2]:
from rdflib import Graph, URIRef
sub = Graph(); sub.bind("", O); sub.bind("skos", "http://www.w3.org/2004/02/skos/core#")
for t in ds.graph(G_REL).triples((URIRef(O + "Татьяна"), None, None)):
    sub.add(t)
print(sub.serialize(format="turtle"))

@prefix : <http://example.org/onegin#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .

:Татьяна a :ГлавныйГерой ;
    rdfs:label "Татьяна Ларина"@ru ;
    :живёт :Деревня,
        :Петербург ;
    :мать :Ларина ;
    :муж :Князь ;
    :няня :Филипьевна ;
    :сестра :Ольга ;
    skos:altLabel "Таня"@ru,
        "Татьяна"@ru .




## Шаг 2. Первый запрос

Запрос SPARQL состоит из **префиксов**, формы ответа и **шаблона** в `WHERE`. Шаблон похож на тройку, только на месте неизвестного стоит **переменная** с вопросительным знаком. Машина запросов ищет в графе все тройки, которые подходят под шаблон, и возвращает значения переменных.

Функция `q` из модуля `onegin_kg` подставляет префиксы и печатает ответ таблицей. Сами префиксы такие:

In [3]:
print(PREFIXES)

PREFIX :     <http://example.org/onegin#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
PREFIX g:    <http://example.org/graph/>



Кто сестра Татьяны? В графе записано `:Ольга :сестра :Татьяна`, шаблон `?кто :сестра :Татьяна` совпадает с этой тройкой при `?кто = :Ольга`.

In [4]:
q(ds, "SELECT ?кто WHERE { ?кто :сестра :Татьяна . }")

,кто
0,:Ольга


Переменная может стоять и на месте предиката. Тогда запрос перечисляет все свойства узла. `LIMIT` ограничивает число строк.

In [5]:
q(ds, "SELECT ?свойство ?значение WHERE { :Татьяна ?свойство ?значение . } LIMIT 12")

,свойство,значение
0,rdf:type,:ГлавныйГерой
1,rdfs:label,Татьяна Ларина
2,:живёт,:Деревня
3,:живёт,:Петербург
4,:мать,:Ларина
5,:муж,:Князь
6,:няня,:Филипьевна
7,:сестра,:Ольга
8,skos:altLabel,Таня
9,skos:altLabel,Татьяна


## Шаг 3. Шаблон из нескольких троек

Несколько троек в `WHERE` образуют **базовый шаблон графа**. Тройки соединены общими переменными: значение `?сестра` в первой тройке должно совпасть со значением `?сестра` во второй. Это то же соединение, что `JOIN` в SQL. Порядок троек на ответ не влияет.

Кто был секундантом у жениха сестры Татьяны?

In [6]:
q(ds, '''
SELECT ?сестра ?жених ?секундант WHERE {
    :Татьяна   :сестра    ?сестра .
    ?жених     :жених     ?сестра .
    ?секундант :секундант ?жених .
}''')

,сестра,жених,секундант
0,:Ольга,:Ленский,:Зарецкий


Запрос прошёл три ребра: Татьяна, Ольга, Ленский, Зарецкий. В тексте романа эти три факта стоят в разных главах, и поиск по словам их не соединит.

## Шаг 4. Литералы, язык метки, FILTER

Имя героя есть литерал. Литерал с меткой языка `"Таня"@ru` и строка `"Таня"` без метки суть **разные значения**. Запрос без метки языка ничего не находит:

In [7]:
q(ds, 'SELECT ?герой WHERE { ?герой skos:altLabel "Таня" . }')

,герой


In [8]:
q(ds, 'SELECT ?герой ?метка WHERE { ?герой skos:altLabel "Таня"@ru ; rdfs:label ?метка . }')

,герой,метка
0,:Татьяна,Татьяна Ларина


`FILTER` оставляет только те решения, для которых условие истинно. Внутри доступны сравнения, логика `&&`, `||`, `!` и функции над строками: `STR` даёт строку без метки языка, `LCASE` переводит в нижний регистр, `STRSTARTS`, `CONTAINS`, `REGEX` проверяют строку.

Все имена, которые начинаются на «та»:

In [9]:
q(ds, 'SELECT ?герой ?имя WHERE { ?герой skos:altLabel ?имя . FILTER(STRSTARTS(LCASE(STR(?имя)), "та")) }')

,герой,имя
0,:Татьяна,Таня
1,:Татьяна,Татьяна


Условие на число и сортировка: события начиная с шестой главы. Строфы тоже имеют номер главы, поэтому они исключены по классу.

In [10]:
q(ds, '''
SELECT ?событие ?глава WHERE {
    ?событие :глава ?глава ; a ?класс .
    FILTER(?глава >= 6 && ?класс != :Строфа)
} ORDER BY ?глава ?событие''')

,событие,глава
0,:вызов,6
1,:гибель_Ленского,6
2,:дуэль,6
3,:приезд_в_Москву,7
4,:встреча_в_Петербурге,8
5,:письмо_Онегина,8
6,:последнее_объяснение,8


## Шаг 5. Необязательная часть: OPTIONAL

Базовый шаблон требует, чтобы совпали все тройки. Если у героя нет мужа, строка с ним пропадёт. `OPTIONAL` делает часть шаблона необязательной: строка остаётся, переменная остаётся пустой.

In [11]:
q(ds, "SELECT ?герой ?муж WHERE { ?герой a :ГлавныйГерой . OPTIONAL { ?герой :муж ?муж } }")

,герой,муж
0,:Ольга,
1,:Ленский,
2,:Татьяна,:Князь
3,:Онегин,


## Шаг 6. Альтернатива: UNION и BIND

`UNION` объединяет решения двух шаблонов. Удобно, когда связь записана в одну сторону и нужны обе стороны. `BIND` записывает значение выражения в новую переменную.

Все связи Ленского с персонажами, в обе стороны:

In [12]:
q(ds, '''
SELECT ?направление ?связь ?кто WHERE {
    { :Ленский ?связь ?кто . BIND("от Ленского" AS ?направление) }
    UNION
    { ?кто ?связь :Ленский . BIND("к Ленскому" AS ?направление) }
    ?кто a ?класс . ?класс rdfs:subClassOf :Персонаж .
}''')

,направление,связь,кто
0,от Ленского,:друг,:Онегин
1,от Ленского,:жених,:Ольга
2,к Ленскому,:секундант,:Зарецкий
3,к Ленскому,:друг,:Онегин


## Шаг 7. Отсутствие и открытый мир

`FILTER NOT EXISTS { ... }` оставляет решения, для которых шаблон внутри не находится. `MINUS { ... }` вычитает решения с общими переменными. Для простых случаев они дают одно и то же.

Какие персонажи ни разу не упомянуты в строфах?

In [13]:
q(ds, '''
SELECT ?герой WHERE {
    ?герой a ?класс . ?класс rdfs:subClassOf :Персонаж .
    FILTER NOT EXISTS { ?герой :упомянут ?строфа }
}''')

,герой
0,:Автор
1,:Дядя
2,:Ларина


Ответ говорит о графе, о романе он ничего не говорит. Ларина в романе упомянута, но правило извлечения её не ищет: в главе 8 «Лариной» названа Татьяна. Граф знает только записанное. Это **открытый мир** из лекции 2: отсутствие тройки значит «неизвестно».

`MINUS`: кто упомянут в главе 6 и не упомянут в главе 8?

In [14]:
q(ds, '''
SELECT DISTINCT ?герой WHERE {
    ?герой :упомянут ?с . ?с :глава 6 .
    MINUS { ?герой :упомянут ?с8 . ?с8 :глава 8 }
}''')

,герой
0,:Зарецкий
1,:Ольга
2,:Гильо


## Шаг 8. Счёт: COUNT, GROUP BY, HAVING

Агрегаты считают по группам. `GROUP BY` задаёт группы, `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` считают внутри группы, `AS` даёт имя результату, `HAVING` фильтрует группы. `ORDER BY DESC(...)` сортирует по убыванию.

Сколько строф упоминает каждого героя:

In [15]:
q(ds, "SELECT ?герой (COUNT(?с) AS ?строф) WHERE { ?герой :упомянут ?с } GROUP BY ?герой ORDER BY DESC(?строф)")

,герой,строф
0,:Татьяна,102
1,:Онегин,92
2,:Ленский,43
3,:Ольга,28
4,:Филипьевна,13
5,:Зарецкий,7
6,:Гильо,2
7,:Князь,2


В каких главах Ленский упомянут не меньше чем в пяти строфах:

In [16]:
q(ds, '''
SELECT ?глава (COUNT(?с) AS ?строф) WHERE {
    :Ленский :упомянут ?с . ?с :глава ?глава .
} GROUP BY ?глава HAVING (COUNT(?с) >= 5) ORDER BY ?глава''')

,глава,строф
0,2,5
1,3,5
2,4,6
3,5,8
4,6,14


## Шаг 9. Пути по свойствам

Путь по свойствам заменяет цепочку троек одним выражением на месте предиката.

| запись | смысл | пример |
|---|---|---|
| `p/q` | сначала `p`, потом `q` | `:сестра/^:жених` |
| `^p` | `p` в обратную сторону | `^:привелоК` |
| `p\|q` | `p` или `q` | `:сестра\|:мать` |
| `p+` | один или больше шагов `p` | `:привелоК+` |
| `p*` | ноль или больше шагов `p` | `rdfs:subClassOf*` |

К чему привели именины Татьяны? Связь `:привелоК` записана только между соседними событиями, `+` проходит всю цепочку:

In [17]:
q(ds, "SELECT ?следствие WHERE { :именины :привелоК+ ?следствие }")

,следствие
0,:ссора_на_балу
1,:вызов
2,:дуэль
3,:гибель_Ленского


**Мини-практика из программы курса**: найти всё, от чего транзитивно зависит узел. Что привело к гибели Ленского? Тот же путь в обратную сторону:

In [18]:
q(ds, "SELECT ?причина WHERE { :гибель_Ленского ^:привелоК+ ?причина }")

,причина
0,:дуэль
1,:вызов
2,:ссора_на_балу
3,:именины


Последовательность: жених сестры Татьяны одним путём. Альтернатива: вся ближняя родня Татьяны.

In [19]:
q(ds, "SELECT ?x WHERE { :Татьяна :сестра/^:жених ?x }")

,x
0,:Ленский


In [20]:
q(ds, "SELECT ?x WHERE { :Татьяна (:сестра|:мать|:муж|:няня) ?x }")

,x
0,:Ольга
1,:Ларина
2,:Князь
3,:Филипьевна


Все события до письма Онегина. Свойство `:следует` связывает событие с предыдущим:

In [21]:
q(ds, "SELECT ?раньше WHERE { :письмо_Онегина :следует+ ?раньше }")

,раньше
0,:встреча_в_Петербурге
1,:приезд_в_Москву
2,:гибель_Ленского
3,:дуэль
4,:вызов
5,:ссора_на_балу
6,:именины
7,:сон_Татьяны
8,:объяснение_в_саду
9,:письмо_Татьяны


Путь отвечает на вопрос «достижим ли узел». Саму цепочку рёбер путь не возвращает. Если нужна цепочка, её собирают запросом с промежуточными переменными или кодом.

## Шаг 10. Классы и подсвойства без машины вывода

В графе записано `:вызов a :Письмо` и `:Письмо rdfs:subClassOf :Событие`. Тройки `:вызов a :Событие` в графе нет. Машина запросов вывода не делает, поэтому прямой запрос находит только события, записанные классом `:Событие`:

In [22]:
q(ds, "SELECT (COUNT(?e) AS ?событий) WHERE { ?e a :Событие }")

,событий
0,5


Путь `a/rdfs:subClassOf*` проходит иерархию классов сам:

In [23]:
q(ds, "SELECT ?событие ?класс WHERE { ?событие a ?класс . ?класс rdfs:subClassOf* :Событие . }")

,событие,класс
0,:приезд_в_Москву,:Событие
1,:сон_Татьяны,:Событие
2,:гибель_Ленского,:Событие
3,:дуэль,:Событие
4,:ссора_на_балу,:Событие
5,:последнее_объяснение,:Объяснение
6,:объяснение_в_саду,:Объяснение
7,:вызов,:Письмо
8,:письмо_Татьяны,:Письмо
9,:письмо_Онегина,:Письмо


Так же с иерархией свойств. Сестра, мать, муж и жених объявлены подсвойствами `:родня`:

In [24]:
q(ds, "SELECT ?связь ?кто WHERE { ?связь rdfs:subPropertyOf :родня . :Татьяна ?связь ?кто . }")

,связь,кто
0,:мать,:Ларина
1,:муж,:Князь
2,:сестра,:Ольга


Хранилища умеют вывод по RDFS и OWL как режим работы. В туториале вывод записан в запросе явно: так видно, откуда берётся каждая строка ответа.

## Шаг 11. Окрестность на два шага

**Мини-практика лекции 7**: окрестность сущности на два шага. Шаблон идёт по любому ребру в любую сторону, кроме `rdf:type`. `GRAPH g:relations` ограничивает обход ручной разметкой, без строф. `VALUES` задаёт начальный узел списком: в список можно поставить несколько героев.

In [25]:
q(ds, '''
SELECT ?старт (COUNT(DISTINCT ?сосед) AS ?соседей) (GROUP_CONCAT(DISTINCT STRAFTER(STR(?сосед), "#"); separator=", ") AS ?кто) WHERE {
    VALUES ?старт { :Зарецкий :Филипьевна }
    GRAPH g:relations {
        { ?старт ?p1 ?через } UNION { ?через ?p1 ?старт }
        { ?через ?p2 ?сосед } UNION { ?сосед ?p2 ?через }
    }
    ?сосед a ?класс . ?класс rdfs:subClassOf :Персонаж .
    FILTER(?p1 != rdf:type && ?p2 != rdf:type && ?сосед != ?старт)
} GROUP BY ?старт''')

,старт,соседей,кто
0,:Зарецкий,4,"Онегин, Ленский, Ольга, Гильо"
1,:Филипьевна,3,"Ларина, Ольга, Князь"


За два шага от Зарецкого лежат все участники дуэли и невеста Ленского. По тексту романа такую окрестность не собрать: в строфах о Зарецком нет имени Ольги.

## Шаг 12. ASK и CONSTRUCT

`ASK` отвечает «да» или «нет»: есть ли в графе решение шаблона. Это проверка утверждения. Языковая модель сказала «Ленский был женихом Татьяны»? Граф отвечает:

In [26]:
print("Онегин друг Ленского:", q(ds, "ASK { :Онегин :друг :Ленский }"))
print("Ленский жених Татьяны:", q(ds, "ASK { :Ленский :жених :Татьяна }"))
print("Ленский жених Ольги:", q(ds, "ASK { :Ленский :жених :Ольга }"))

Онегин друг Ленского: True
Ленский жених Татьяны: False
Ленский жених Ольги: True


Ответ «нет» здесь значит «в графе такого нет». Для проверки ответа модели этого достаточно, если граф полон по этому свойству. Этот контур проверки вернётся в лекции 8.

`CONSTRUCT` строит **новый граф** по шаблону. Например, связь «знакомы»: два героя участвовали в одном событии.

In [27]:
known = q(ds, '''
CONSTRUCT { ?a :знакомы ?b } WHERE {
    ?событие :участник ?a , ?b .
    FILTER(STR(?a) < STR(?b))
}''')
print("новых троек:", len(known))
print(known.serialize(format="turtle"))

новых троек: 17
@prefix : <http://example.org/onegin#> .

:Гильо :знакомы :Зарецкий,
        :Ленский,
        :Онегин .

:Князь :знакомы :Онегин,
        :Татьяна .

:Ларина :знакомы :Ленский,
        :Ольга,
        :Онегин,
        :Татьяна .

:Зарецкий :знакомы :Ленский,
        :Онегин .

:Ольга :знакомы :Онегин,
        :Татьяна .

:Ленский :знакомы :Ольга,
        :Онегин,
        :Татьяна .

:Онегин :знакомы :Татьяна .




Есть ещё форма `DESCRIBE :узел`: она возвращает описание узла, и его состав определяет хранилище.

## Шаг 13. Именованные графы и происхождение

Набор данных состоит из нескольких графов, у каждого своё имя. `GRAPH ?граф { ... }` говорит, в каком графе нашлась тройка. Запрос без `GRAPH` в этом блокноте видит объединение графов.

In [28]:
q(ds, "SELECT ?граф (COUNT(*) AS ?троек) WHERE { GRAPH ?граф { ?s ?p ?o } } GROUP BY ?граф")

,граф,троек
0,g:relations,269
1,g:text,2453


Откуда граф знает об Ольге:

In [29]:
q(ds, "SELECT ?граф ?свойство (COUNT(?o) AS ?значений) WHERE { GRAPH ?граф { :Ольга ?свойство ?o } } GROUP BY ?граф ?свойство ORDER BY ?граф")

,граф,свойство,значений
0,g:relations,rdf:type,1
1,g:relations,rdfs:label,1
2,g:relations,:мать,1
3,g:relations,:сестра,1
4,g:relations,skos:altLabel,1
5,g:text,:упомянут,28


Сестра и мать записаны человеком. Упоминания в 28 строфах извлечены программой и могут содержать ошибки извлечения. Агент, который ссылается на факт, может назвать и его происхождение.

## Шаг 14. Граф и поиск по словам на одном вопросе

Вопрос из выборки занятия 5: «ответ Онегина на письмо Татьяны». Нужные строфы: 4.XII–4.XVI, объяснение в саду. Поиск BM25F из занятия 5:

In [30]:
import csv
from search_bm25 import Pipeline, Index
rows = list(csv.DictReader(open("data/onegin_stanzas.csv", encoding="utf-8")))
idx = Index([{"название": r["название"], "текст": r["текст"]} for r in rows], Pipeline(),
            {"название": 2.0, "текст": 1.0}, [r["id"] for r in rows])
need = {"4.XII", "4.XIII", "4.XIV", "4.XV", "4.XVI"}
top = [d for d, _ in idx.bm25f("ответ Онегина на письмо Татьяны", k=5)]
print("BM25F, первые пять:", top)
print("нужных строф среди них:", len(need & set(top)))

BM25F, первые пять: ['8.XXXIII', '3.XXXVI', '3.ПТ', '3.XXXI', '8.XXI']
нужных строф среди них: 0


Слова «письмо», «Татьяна», «Онегин» есть в самом письме и в строфах о нём. В строфах ответа их почти нет. По графу ответ находится связью `:ответНа`:

In [31]:
q(ds, '''
SELECT ?строфа ?перваяСтрока WHERE {
    ?ответ :ответНа ?письмо .
    ?письмо :автор :Татьяна .
    ?ответ :вСтрофах ?с .
    ?с rdfs:label ?строфа ; :перваяСтрока ?перваяСтрока .
} ORDER BY ?строфа''')

,строфа,перваяСтрока
0,4.XII,"Минуты две они молчали,"
1,4.XIII,Когда бы жизнь домашним кругом
2,4.XIV,Но я не создан для блаженства;
3,4.XV,Что может быть на свете хуже
4,4.XVI,Мечтам и годам нет возврата;


SPARQL умеет и искать по тексту, но без ранжирования и без нормализации слов: `CONTAINS` проверяет подстроку.

In [32]:
q(ds, 'SELECT (COUNT(?с) AS ?строф) WHERE { ?с :текст ?t . FILTER(CONTAINS(LCASE(?t), "письм")) }')

,строф
0,18


Восемнадцать строф без порядка. Каналы делят работу: поиск по словам находит строфу по её словам, граф находит строфу по связи. Как соединить их в одной выдаче, разбирает лекция 8.

## Шаг 15. Тот же приём на положении о рецензировании

Граф положения о рецензировании из модуля 1 (`data/review.ttl`, копия из лекции 2) хранит связи «зависит от» между пунктами, разделами и терминами. От чего прямо и транзитивно зависит пункт 6.5 «Принятие решения»? У графа свой префикс.

In [33]:
from rdflib import Graph
review = Graph().parse("data/review.ttl")
RP = "PREFIX : <http://example.org/review.owl#>\nPREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n"
direct = q(review, "SELECT ?название WHERE { :R65 :зависит_от ?x . ?x rdfs:label ?название } ORDER BY ?название", prefixes=RP)
closure = q(review, "SELECT ?название WHERE { :R65 :зависит_от+ ?x . ?x rdfs:label ?название } ORDER BY ?название", prefixes=RP)
print("прямо:", len(direct), " транзитивно:", len(closure))
closure

прямо: 4  транзитивно: 14


,название
0,6.1 Число рецензий
1,6.3 Содержание рецензии
2,6.4 Ответ на замечания
3,Автор
4,Замечание
5,Ответ авторов
6,Раздел 1. Область применения
7,Раздел 3. Термины и определения
8,Раздел 6. Порядок рецензирования
9,Редакционная коллегия


Прямых зависимостей четыре, вместе с транзитивными четырнадцать. Этот вопрос редактора в лекции 1 остался без ответа: поиск по словам связей не видит.

## Шаг 16. Точка доступа SPARQL

В блокноте граф лежит в памяти процесса Python. В работе граф хранят в **хранилище троек** и запрашивают через **точку доступа**: адрес HTTP, который принимает запрос SPARQL и возвращает ответ в JSON, CSV или RDF. Два открытых хранилища:

```bash
# Oxigraph: один исполняемый файл
pip install pyoxigraph            # библиотека для Python
oxigraph load --location store --file data/onegin_relations.ttl
oxigraph serve --location store   # точка доступа http://localhost:7878/query

# Apache Jena Fuseki: сервер на Java
fuseki-server --file=data/onegin_relations.ttl /onegin   # http://localhost:3030/onegin/sparql
```

Запрос к точке доступа из Python:

```python
import requests
r = requests.get("http://localhost:7878/query",
                 params={"query": PREFIXES + "SELECT ?кто WHERE { ?кто :сестра :Татьяна }"},
                 headers={"Accept": "application/sparql-results+json"})
r.json()["results"]["bindings"]
```

Ячейки этого шага не выполняются: для них нужен запущенный сервер.

## Шаг 17. SPARQL как инструмент агента

Агент из модуля 1 вызывает инструменты по описанию. Инструмент запроса к графу возвращает таблицу или понятную ошибку, и ограничивает размер ответа, чтобы не переполнить контекст модели.

In [34]:
import json

SPARQL_TOOL = {
    "name": "sparql_query",
    "description": ("Запрос SPARQL 1.1 к графу «Евгения Онегина». Префикс : уже задан. "
                    "Классы: :Персонаж, :Событие, :Строфа. Свойства: :сестра, :жених, :друг, :муж, "
                    ":участник, :привелоК, :ответНа, :вСтрофах, :упомянут. Возвращает не больше 20 строк."),
    "parameters": {"query": "string"},
}

def sparql_query(query, limit=20):
    try:
        res = q(ds, query)
    except Exception as e:                       # синтаксическая ошибка уходит модели текстом
        return json.dumps({"ошибка": str(e)[:200]}, ensure_ascii=False)
    if isinstance(res, bool):
        return json.dumps({"ответ": res}, ensure_ascii=False)
    return json.dumps({"строк": len(res), "строки": res.head(limit).to_dict("records")}, ensure_ascii=False)

print(sparql_query("SELECT ?причина WHERE { :гибель_Ленского ^:привелоК+ ?причина }"))
print(sparql_query("SELECT ?x WHERE { :Ленский :жених ?x"))

{"строк": 4, "строки": [{"причина": ":дуэль"}, {"причина": ":вызов"}, {"причина": ":ссора_на_балу"}, {"причина": ":именины"}]}
{"ошибка": "Expected {SelectQuery | ConstructQuery | DescribeQuery | AskQuery}, found end of text  (at char 332), (line:7, col:37)"}


Протокол вызова есть доказательство ответа: агент может сослаться на запрос и строки, которые он вернул.

## Упражнения

1. Кто участвовал в дуэли? Выведите метки участников.
2. Найдите все письма в графе вместе с автором и адресатом.
3. Какие события произошли в Петербурге? Отсортируйте по главе.
4. Сколько строф каждой главы упоминают Татьяну?
5. В каких строфах упомянуты одновременно Татьяна и Ольга?
6. Какие события прямо или через цепочку привели к последнему объяснению Татьяны?
7. Проверьте запросом `ASK` утверждение «Зарецкий был секундантом Онегина».
8. Постройте запросом `CONSTRUCT` связь «соседи»: два персонажа живут в одном месте.

Ответы ниже. Сначала попробуйте сами.

### Ответы

In [35]:
q(ds, "SELECT ?участник ?метка WHERE { :дуэль :участник ?участник . ?участник rdfs:label ?метка }")

,участник,метка
0,:Гильо,"Гильо, слуга Онегина"
1,:Зарецкий,Зарецкий
2,:Ленский,Владимир Ленский
3,:Онегин,Евгений Онегин


In [36]:
q(ds, "SELECT ?письмо ?автор ?адресат WHERE { ?письмо a :Письмо ; :автор ?автор ; :адресат ?адресат }")

,письмо,автор,адресат
0,:вызов,:Ленский,:Онегин
1,:письмо_Татьяны,:Татьяна,:Онегин
2,:письмо_Онегина,:Онегин,:Татьяна


In [37]:
q(ds, "SELECT ?событие ?глава WHERE { ?событие :место :Петербург ; :глава ?глава } ORDER BY ?глава")

,событие,глава
0,:встреча_в_Петербурге,8
1,:последнее_объяснение,8
2,:письмо_Онегина,8


In [38]:
q(ds, "SELECT ?глава (COUNT(?с) AS ?строф) WHERE { :Татьяна :упомянут ?с . ?с :глава ?глава } GROUP BY ?глава ORDER BY ?глава")

,глава,строф
0,2,3
1,3,20
2,4,7
3,5,26
4,6,3
5,7,28
6,8,15


In [39]:
q(ds, "SELECT ?строфа WHERE { :Татьяна :упомянут ?с . :Ольга :упомянут ?с . ?с rdfs:label ?строфа } ORDER BY ?строфа")

,строфа
0,2.XXVII
1,3.V
2,3.XXXVI
3,3.XXXVII
4,4.XLIX
5,4.XLVIII
6,5.XLII
7,5.XX
8,5.XXI
9,5.XXXVI


In [40]:
q(ds, "SELECT ?причина WHERE { :последнее_объяснение ^:привелоК+ ?причина }")

,причина
0,:письмо_Онегина
1,:встреча_в_Петербурге


In [41]:
print(q(ds, "ASK { :Зарецкий :секундант :Онегин }"), "| секундант Онегина:", q(ds, "SELECT ?x WHERE { ?x :секундант :Онегин }")["x"].tolist())

False | секундант Онегина: [':Гильо']


In [42]:
same = q(ds, "CONSTRUCT { ?a :соседи ?b } WHERE { ?a :живёт ?м . ?b :живёт ?м . FILTER(STR(?a) < STR(?b)) }")
print(same.serialize(format="turtle"))

@prefix : <http://example.org/onegin#> .

:Ленский :соседи :Онегин,
        :Татьяна .

:Онегин :соседи :Татьяна .




## Шпаргалка

```sparql
PREFIX : <http://example.org/onegin#>
SELECT DISTINCT ?x ?y (COUNT(?z) AS ?n)     # форма ответа; ASK, CONSTRUCT, DESCRIBE
WHERE {
  ?x :p ?y ; :q ?z .                         # шаблон: ; тот же субъект, , тот же предикат
  ?y a/rdfs:subClassOf* :Класс .             # путь: / ^ | + *
  OPTIONAL { ?x :r ?w }                      # необязательная часть
  { ?x :s ?v } UNION { ?v :s ?x }            # альтернатива
  FILTER(?n > 2 && LANG(?l) = "ru")          # условие
  FILTER NOT EXISTS { ?x :t ?u }             # отсутствие в графе
  BIND(STR(?l) AS ?строка)                   # новая переменная
  VALUES ?x { :a :b }                        # список значений
  GRAPH ?g { ?x :p ?y }                      # именованный граф
}
GROUP BY ?x ?y HAVING (COUNT(?z) > 1)
ORDER BY DESC(?n) LIMIT 10 OFFSET 0
```

**Типичные ошибки.** Строка без метки языка не равна строке с меткой. Переменная, которой нет в `GROUP BY`, не может стоять в ответе без агрегата. Пустой ответ значит «в графе не записано». Запрос без вывода не видит подклассов: нужен путь `rdfs:subClassOf*`.